# Vacinas ao longo do tempo

**Projeto:** PyAnalytics — UFSC Araranguá  
**Sprint 3** — `notebooks/Sprint_3_vacinas_no_tempo`  
**Base:** `data/processed/vacinas_analitico.csv`

## Contexto e motivação

A vacinação no município não acontece de forma igual todos os meses. Campanhas, rotina e sazonalidade mudam o volume de aplicações. Este notebook ajuda a **enxergar mudanças, picos e períodos fora do habitual**, de forma simples, para apoiar a leitura cidadã dos dados.

> O que mostramos aqui é o **volume de aplicações registradas** no consolidado.  
> **Não** é cobertura vacinal da população, **não** é diagnóstico de surto e **não** mede sozinho a “necessidade de saúde”.


## Perguntas que orientam a exploração

1. Como o volume de aplicações varia ao longo do período?
2. Em quais meses houve aumentos ou reduções relevantes?
3. O pico coincide com alguma vacina específica (ex.: influenza)?
4. Como vacinas, salas (unidades/bairros) e estratégias se comportam no tempo?
5. Janeiro aparece no filtro, mas quase não tem aplicação — isso fica visível?


## 1. Preparação dos dados


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.figsize": (11, 5),
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "legend.fontsize": 9,
})

CAMINHOS = [
    Path("../../data/processed/vacinas_analitico.csv"),
    Path("../data/processed/vacinas_analitico.csv"),
    Path("data/processed/vacinas_analitico.csv"),
]
caminho = next(p for p in CAMINHOS if p.exists())
df = pd.read_csv(caminho, low_memory=False)
print(f"Arquivo: {caminho.resolve()}")
print(f"Registros: {len(df):,}".replace(",", "."))

In [ ]:
def padronizar_estrategia(serie: pd.Series) -> pd.Series:
    """Unifica rótulos parecidos de estratégia."""
    s = serie.astype("string").str.strip()
    return s.replace({
        "Rotina (1)": "Rotina",
        "Pós": "Pós-exposição",
        "Monitoramento das Estratégias de Vacinação": "Monitoramento",
    })


def preparar_datas(frame: pd.DataFrame) -> pd.DataFrame:
    """Cria data completa e mês (MM/AAAA) sem perder rastreabilidade."""
    out = frame.copy()
    out["data_aplicacao_bruta"] = out["data_da_aplicacao"]
    out["data_aplicacao"] = pd.to_datetime(
        out["data_da_aplicacao"], dayfirst=True, errors="coerce"
    )
    out["mes_periodo"] = out["data_aplicacao"].dt.to_period("M")
    out["mes_label"] = out["data_aplicacao"].dt.strftime("%m/%Y")
    out["dia_label"] = out["data_aplicacao"].dt.strftime("%d/%m/%Y")
    return out


df["estrategia"] = padronizar_estrategia(df["estrategia"])
df = preparar_datas(df)

# Sala de vacinação no consolidado = unidade/bairro de aplicação
df["sala"] = df["nome_do_bairro"].astype("string").str.strip()

com_data = df["data_aplicacao"].notna().sum()
sem_data = df["data_aplicacao"].isna().sum()
print("Com data válida:", f"{com_data:,}".replace(",", "."))
print("Sem data (mantidos no total, fora da série temporal):", f"{sem_data:,}".replace(",", "."))
print("Exemplo de rastreabilidade:")
display(df[["data_aplicacao_bruta", "dia_label", "mes_label"]].dropna().head(3))


### Período disponível na base

Os registros com data válida cobrem **fevereiro a junho de 2026**.  
**Janeiro/2026 não aparece nas aplicações** deste consolidado (zero registros). Se algum relatório/filtro listar janeiro, isso precisa ficar explícito: o mês entra no calendário, mas **não há volume para analisar**.


In [ ]:
serie = df.dropna(subset=["data_aplicacao"]).copy()

inicio = serie["mes_periodo"].min()
fim = serie["mes_periodo"].max()
calendario = pd.period_range(inicio, fim, freq="M")

mensal = (
    serie.groupby("mes_periodo")
    .size()
    .reindex(calendario, fill_value=0)
    .rename("aplicacoes")
    .to_frame()
)
mensal["mes_label"] = mensal.index.strftime("%m/%Y")
mensal["variacao_pct"] = mensal["aplicacoes"].pct_change() * 100

# Janeiro do mesmo ano entra com 0 para ficar visível no filtro/visão
jan = pd.Period(f"{inicio.year}-01", freq="M")
if jan not in mensal.index and jan < inicio:
    linha_jan = pd.DataFrame(
        {"aplicacoes": [0], "mes_label": [jan.strftime("%m/%Y")], "variacao_pct": [np.nan]},
        index=pd.PeriodIndex([jan]),
    )
    mensal = pd.concat([linha_jan, mensal]).sort_index()
    mensal["variacao_pct"] = mensal["aplicacoes"].pct_change() * 100

mensal_view = mensal[["mes_label", "aplicacoes", "variacao_pct"]].copy()
mensal_view["variacao_pct"] = mensal_view["variacao_pct"].replace([np.inf, -np.inf], np.nan)
display(mensal_view.round(1))
print("Janeiro no gráfico:", "sim (0 aplicações)" if jan in mensal.index else "não")


## 2. Como o volume varia ao longo do período?


In [ ]:
def anotar_variacoes(ax, x, y, variacao, deslocamento=18):
    """Coloca % de aumento/redução quando o valor é finito."""
    for xi, yi, v in zip(x, y, variacao):
        if pd.isna(v) or not np.isfinite(v):
            continue
        cor = "#1b7f4a" if v >= 0 else "#b00020"
        ax.annotate(
            f"{v:+.0f}%",
            xy=(xi, yi),
            xytext=(0, deslocamento if v >= 0 else -deslocamento),
            textcoords="offset points",
            ha="center",
            va="center",
            fontsize=9,
            fontweight="bold",
            color=cor,
        )


fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(mensal))
ax.plot(x, mensal["aplicacoes"], marker="o", linewidth=2.2, color="#1f4e79")
cores = ["#1b7f4a" if v >= 0 else "#b00020" for v in mensal["aplicacoes"].diff().fillna(0)]
ax.scatter(x, mensal["aplicacoes"], c=cores, s=70, zorder=3)
anotar_variacoes(ax, x, mensal["aplicacoes"].values, mensal["variacao_pct"].values)

# Janeiro → fevereiro: destaca início sem usar % infinita
if mensal.iloc[0]["aplicacoes"] == 0 and len(mensal) > 1 and mensal.iloc[1]["aplicacoes"] > 0:
    ax.annotate(
        "início dos registros",
        xy=(1, mensal.iloc[1]["aplicacoes"]),
        xytext=(0, 28),
        textcoords="offset points",
        ha="center",
        fontsize=8,
        color="#555555",
    )

ax.set_xticks(x)
ax.set_xticklabels(mensal["mes_label"])
ax.set_title("Quantidade de aplicações de vacinas por mês")
ax.set_xlabel("Mês")
ax.set_ylabel("Número de aplicações")
ax.set_ylim(0, max(mensal["aplicacoes"].max() * 1.18, 1))
plt.tight_layout()
plt.show()


### Leitura (geral)

- O volume **sobe forte até abril** e depois **recua em maio e junho**.
- As variações percentuais aparecem **nos aumentos e nas reduções**.
- Janeiro fica com **0 aplicações**, deixando claro que o mês não tem movimento na base.

Isso descreve **ritmo de registro/aplicação**, não cobertura da cidade nem surto.


## 3. O pico coincide com a influenza?


In [ ]:
serie["grupo_vacina"] = np.where(
    serie["nome_da_vacina"].astype(str).str.contains("influ", case=False, na=False),
    "Influenza trivalente",
    "Demais vacinas",
)

por_grupo = (
    serie.groupby(["mes_periodo", "grupo_vacina"])
    .size()
    .unstack(fill_value=0)
    .reindex(mensal.index, fill_value=0)
)

fig, ax = plt.subplots(figsize=(11, 5))
for col, cor in zip(por_grupo.columns, ["#c0392b", "#2c3e50"]):
    ax.plot(
        range(len(por_grupo)),
        por_grupo[col],
        marker="o",
        linewidth=2.2,
        label=col,
        color=cor,
    )
ax.set_xticks(range(len(por_grupo)))
ax.set_xticklabels(mensal["mes_label"])
ax.set_title("Aplicações por mês: influenza × demais vacinas")
ax.set_xlabel("Mês")
ax.set_ylabel("Número de aplicações")
ax.legend(title="Grupo de vacina")
plt.tight_layout()
plt.show()

pico_mes = mensal["aplicacoes"].idxmax()
total_pico = int(mensal.loc[pico_mes, "aplicacoes"])
infl_pico = int(por_grupo.loc[pico_mes].get("Influenza trivalente", 0))
share = (infl_pico / total_pico * 100) if total_pico else 0
print(f"Mês de maior volume: {pico_mes.strftime('%m/%Y')} ({total_pico:,} aplicações)".replace(",", "."))
print(f"Influenza nesse mês: {infl_pico:,} ({share:.1f}% do total)".replace(",", "."))

### Leitura (influenza)

O aumento de **abril** acompanha a **influenza trivalente**, típica de campanha sazonal.  
As demais vacinas ficam mais estáveis — o pico geral **não** deve ser lido como surto nem como mudança brusca de toda a vacinação de rotina.


## 4. Vacinas no tempo


In [ ]:
def serie_mensal_categoria(frame, coluna, categorias):
    """Série mensal para um conjunto fixo de categorias."""
    base = (
        frame[frame[coluna].isin(categorias)]
        .groupby(["mes_periodo", coluna])
        .size()
        .unstack(fill_value=0)
    )
    return base.reindex(mensal.index, fill_value=0)


def plot_linhas_mensais(tabela, titulo, legenda_titulo):
    """Um gráfico de linhas por figura (sem painéis acoplados)."""
    fig, ax = plt.subplots(figsize=(11, 5))
    for col in tabela.columns:
        ax.plot(range(len(tabela)), tabela[col], marker="o", linewidth=2, label=str(col))
    ax.set_xticks(range(len(tabela)))
    ax.set_xticklabels(mensal["mes_label"])
    ax.set_title(titulo)
    ax.set_xlabel("Mês")
    ax.set_ylabel("Número de aplicações")
    ax.legend(title=legenda_titulo, bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()


top_vacinas = serie["nome_da_vacina"].value_counts().head(5).index.tolist()
tab_top = serie_mensal_categoria(serie, "nome_da_vacina", top_vacinas)
plot_linhas_mensais(
    tab_top,
    "Evolução mensal das 5 vacinas mais aplicadas",
    "Vacina",
)

### Vacinas menos frequentes

Em vez de heatmap vazio ou boxplot com poucos pontos, usamos **um gráfico de barras** com o total no período. Nomes no eixo — sem legenda longa.


In [ ]:
contagem_vacina = serie["nome_da_vacina"].value_counts()
menos_frequentes = contagem_vacina[contagem_vacina < 100].sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, max(4, 0.35 * len(menos_frequentes))))
ax.barh(menos_frequentes.index.astype(str), menos_frequentes.values, color="#5d6d7e")
ax.set_title("Vacinas menos frequentes — total de aplicações no período")
ax.set_xlabel("Número de aplicações")
ax.set_ylabel("Vacina")
for i, v in enumerate(menos_frequentes.values):
    ax.text(v + max(menos_frequentes.values) * 0.01, i, str(v), va="center", fontsize=8)
plt.tight_layout()
plt.show()
print(f"Vacinas com menos de 100 aplicações: {len(menos_frequentes)}")

## 5. Estratégias no tempo


In [ ]:
top_estrategias = serie["estrategia"].value_counts().head(4).index.tolist()
tab_est = serie_mensal_categoria(serie, "estrategia", top_estrategias)
plot_linhas_mensais(
    tab_est,
    "Evolução mensal das principais estratégias",
    "Estratégia",
)

### Leitura (estratégia)

**Rotina** concentra a maior parte do volume. Estratégias pontuais aparecem com volumes menores e não devem ser lidas sozinhas como mudança estrutural da demanda.


## 6. Salas (unidades/bairros) no tempo


In [ ]:
top_salas = serie["sala"].value_counts().head(5).index.tolist()
tab_sala = serie_mensal_categoria(serie, "sala", top_salas)
plot_linhas_mensais(
    tab_sala,
    "Evolução mensal das 5 salas/unidades com mais aplicações",
    "Sala (bairro/unidade)",
)

### Leitura (salas)

No consolidado, a “sala” corresponde ao **bairro/unidade** de aplicação.  
Diferenças entre salas mostram **onde o registro foi maior**, não necessariamente quem tem mais “necessidade” de vacina.


## 7. Síntese objetiva


In [ ]:
primeiro_com_dado = mensal.index[mensal["aplicacoes"] > 0][0]
resumo = pd.DataFrame({
    "Indicador": [
        "Período com data válida",
        "Total de aplicações com data",
        "Mês de maior volume",
        "Variação até o pico (1º mês com dado → pico)",
        "Participação da influenza no pico",
        "Janeiro/2026 na base",
    ],
    "Valor": [
        f"{mensal.index.min().strftime('%m/%Y')} a {mensal.index.max().strftime('%m/%Y')}",
        f"{int(serie.shape[0]):,}".replace(",", "."),
        f"{pico_mes.strftime('%m/%Y')} ({total_pico:,})".replace(",", "."),
        f"{((mensal.loc[pico_mes, 'aplicacoes'] / mensal.loc[primeiro_com_dado, 'aplicacoes'] - 1) * 100):+.0f}%",
        f"{share:.1f}%",
        "0 aplicações (mês sem movimento no consolidado)",
    ],
})
display(resumo)

## O que os resultados podem significar (com cuidado)

| Achado | Interpretação adequada | Evitar ler como |
|--------|------------------------|-----------------|
| Pico em abril | Maior volume de **aplicações registradas**, puxado pela influenza | Surto, epidemia ou emergência |
| Queda maio–junho | Redução do ritmo de aplicações após o pico | “Piorou a saúde” da população |
| Diferença entre salas | Onde houve mais registros no período | Necessidade ou cobertura por bairro |
| Vacinas raras | Poucos registros no consolidado | Conclusão clínica sobre raridade real |

## Critérios de aceite

| Critério | Situação |
|----------|----------|
| Visualizações simples para comparar períodos (linha/barra) | Atendido — sem boxplot, sem log, sem heatmap vazio |
| Exploração por mês, vacina, sala e estratégia | Atendido |
| Datas em mês (`04/2026`) e dia (`09/04/2026`) com rastreabilidade | Atendido (`mes_label`, `dia_label`, `data_aplicacao_bruta`) |
| Janeiro visível como mês sem aplicações | Atendido |
| Interpretação cuidadosa (não trata pico como surto/cobertura) | Atendido |
| Gráficos separados (sem 2–3 acoplados) | Atendido |
| Sem tabelas repetindo o gráfico | Atendido |
| Legendas com nome real (Vacina, Estratégia, Sala) | Atendido |
| % de variação em aumentos **e** reduções | Atendido |
| Poucos gráficos, sem painéis vazios | Atendido |
